# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1

In [4]:
months = ["2026-09", "2026-10"]

revenue_month = np.array([
    amount[clean & (month == m)].sum()
    for m in months
])
print("Сентябрь:", round(revenue_month[0]), "₸")
print("Октябрь:", round(revenue_month[1]), "₸")

change = revenue_month[1] - revenue_month[0]
change_pct = change / revenue_month[0] * 100

print("Изменение:", round(change), "₸")
print(f"Изменение в процентах: {change_pct:.1f}%")

Сентябрь: 35243410 ₸
Октябрь: 34419520 ₸
Изменение: -823890 ₸
Изменение в процентах: -2.3%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2

In [8]:
categories = np.unique(cat[clean])
category_table = np.array([
    [
        amount[clean & (cat == c) & (month == m)].sum()
        for m in months
    ]
    for c in categories
])
print("Строки — категории:", categories)
print("Столбцы — сентябрь, октябрь:")
print(category_table)

for i in range(len(categories)):
    total[i] = category_table[i, 0] + category_table[i, 1]
    change_percent[i] = (
        (category_table[i, 1] - category_table[i, 0])/ category_table[i, 0] * 100
    )

print("Больше всего выручки:", categories[np.argmax(total)])
print("Сильнее всего упала:", categories[np.argmin(change_percent)])
print("Изменение этой категории:",
      round(change_percent[np.argmin(change_percent)], 1), "%")

Строки — категории: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
Столбцы — сентябрь, октябрь:
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки: Электроника
Сильнее всего упала: Одежда
Изменение этой категории: -7.2 %


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3

In [12]:
counts = np.zeros(len(categories))

for i in range(len(categories)):
    mask = clean & (cat == categories[i]) & (month == "2026-10")
    counts[i] = mask.sum()

    average = category_table[i, 1] / counts[i]
    print(categories[i], "средний чек:", round(average), "₸")

average = np.zeros(len(categories))

for i in range(len(categories)):
    average[i] = category_table[i, 1] / counts[i]

print("Самый высокий средний чек в категории:", categories[np.argmax(average)])

Дом средний чек: 10768 ₸
Книги средний чек: 3816 ₸
Красота средний чек: 6159 ₸
Одежда средний чек: 13046 ₸
Электроника средний чек: 33600 ₸
Самый высокий средний чек в категории: Электроника


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

In [15]:
cities = np.unique(city[clean])
city_table = np.zeros((len(cities), 2))

for i in range(len(cities)):
    city_table[i, 0] = amount[clean & (city == cities[i]) & (month == "2026-09")].sum()
    city_table[i, 1] = amount[clean & (city == cities[i]) & (month == "2026-10")].sum()

sep_total = city_table[:, 0].sum()
oct_total = city_table[:, 1].sum()

for i in range(len(cities)):
    sep_share = city_table[i, 0] / sep_total
    oct_share = city_table[i, 1] / oct_total
    print(cities[i], "сентябрь:", round(sep_share * 100, 1), "%; октябрь:",
          round(oct_share * 100, 1), "%")
    
for i in range(len(cities)):
    if cities[i] == "Шымкент":
        sep_share = city_table[i, 0] / sep_total
        oct_share = city_table[i, 1] / oct_total
        print("Изменение доли Шымкента:",
              round((oct_share - sep_share) * 100, 1))

Актобе сентябрь: 7.3 %; октябрь: 7.4 %
Алматы сентябрь: 41.1 %; октябрь: 42.5 %
Астана сентябрь: 25.6 %; октябрь: 25.7 %
Атырау сентябрь: 2.6 %; октябрь: 2.8 %
Караганда сентябрь: 10.4 %; октябрь: 9.8 %
Шымкент сентябрь: 12.9 %; октябрь: 11.7 %
Изменение доли Шымкента: -1.2


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

In [17]:
total_drop = sep_total - oct_total

for i in range(len(cities)):
    city_drop = city_table[i, 0] - city_table[i, 1]
    contribution = city_drop / total_drop * 100
    print(cities[i], "вклад в падение:", round(contribution, 1), "%")
contributions = np.zeros(len(cities))

for i in range(len(cities)):
    city_drop = city_table[i, 0] - city_table[i, 1]
    contributions[i] = city_drop / total_drop * 100

Актобе вклад в падение: 4.4 %
Алматы вклад в падение: -17.2 %
Астана вклад в падение: 22.3 %
Атырау вклад в падение: -5.6 %
Караганда вклад в падение: 33.6 %
Шымкент вклад в падение: 62.5 %


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

In [18]:
for i in range(len(cities)):
    city_total = city_table[i, 0] + city_table[i, 1]

    sep_share = city_table[i, 0] / city_total
    oct_share = city_table[i, 1] / city_total

    print(cities[i], "сентябрь:", round(sep_share * 100, 1), "%; октябрь:",
          round(oct_share * 100, 1), "%")

Актобе сентябрь: 50.4 %; октябрь: 49.6 %
Алматы сентябрь: 49.8 %; октябрь: 50.2 %
Астана сентябрь: 50.5 %; октябрь: 49.5 %
Атырау сентябрь: 48.8 %; октябрь: 51.2 %
Караганда сентябрь: 52.0 %; октябрь: 48.0 %
Шымкент сентябрь: 53.0 %; октябрь: 47.0 %


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8